In [1]:
import cantera as ct
import numpy as np

In [2]:
# Creazione oggetto reactants (miscela metano/aria)
reactants = ct.Solution('gri30.yaml', transport_model="unity-Lewis-number")

In [3]:
# Definizione specie della progress variable
pv_species = ['CO2', 'H2O', 'CO', 'H2'] # oppure, ad es.: ['O2', 'CO2', 'H2O', 'H2'], ['O2']
pv_indices = [reactants.species_index(i) for i in pv_species]

# Personalizzazione dei pesi per le specie della progress variable (opzionale)
# +++++++++ DEVE AVERE LA STESSA DIMENSIONE DI PV_SPECIES +++++++++
custom_weights = None
# species_signs = [-1, 1, 1, 1]
# custom_weights = [100 * sign / reactants.molecular_weights[reactants.species_index(sp)]
#                   for sp, sign in zip(pv_species, species_signs)
#                  ]  # inverso del peso molecolare, con segno; default 'None'

# Definizione dei pesi per le specie della progress variable
pv_weights = np.zeros(reactants.n_species)
if custom_weights is not None:
    if len(custom_weights) != len(pv_species):
        raise ValueError("WARNING: custom_weights must be the same size as pv_species!")
    for idx, w in zip(pv_indices, custom_weights):
        pv_weights[idx] = w
else:
    for idx in pv_indices:
        pv_weights[idx] = 1.0

In [4]:
# Definizione miscela di reagenti
fuel = 'CH4'
air = 'O2:1.0, N2:3.76'
phi = 1

reactants.set_equivalence_ratio(phi, fuel, air)

print(f"Initial mass fractions at phi= {phi}:")
for i in range(reactants.n_species):
    if reactants.Y[i] > 0:
        print(f"- {reactants.species_name(i)}: {reactants.Y[i]:.6f}")


Initial mass fractions at phi= 1:
- O2: 0.220141
- CH4: 0.055187
- N2: 0.724672


In [5]:
# Calcolo della costante specifica della miscela
R_u = ct.gas_constant                 # [J/(kmol*K)]
MW  = reactants.mean_molecular_weight # [kg/kmol]
R_s = R_u / MW                        # [J/(kg*K)]
print(f"Specific gas constant = {R_s:3.9f} [J/(kg*K)]")

Specific gas constant = 300.883587758 [J/(kg*K)]


In [6]:
T_u = 298 # [K]
P_u = ct.one_atm # [Pa]
reactants.TP = T_u, P_u

# Altre proprietà
print(f"\nInitial density           = {reactants.density: .11e} [kg/m^3]") # uguale a 'reactants.density_mass'
print(f"\nInitial dynamic viscosity = {reactants.viscosity: .11e} [Pa*s]")
print(f"\nInitial progress variable = {np.dot(reactants.Y, pv_weights): .11e}")
print(f"\nInitial enthalpy          = {reactants.h: .11e} [J/kg]") # uguale a 'reactants.enthalpy_mass'
print(f"\nInitial heat release rate = {reactants.heat_release_rate: .11e} [W/m^3]")



Initial density           =  1.13006090182e+00 [kg/m^3]

Initial dynamic viscosity =  1.79330006154e-05 [Pa*s]

Initial progress variable =  0.00000000000e+00

Initial enthalpy          = -2.56741249396e+05 [J/kg]

Initial heat release rate = -2.98778304443e-27 [W/m^3]


In [7]:
reactants()


  gri30:

       temperature   298 K
          pressure   1.0132e+05 Pa
           density   1.1301 kg/m^3
  mean mol. weight   27.633 kg/kmol
   phase of matter   gas

                          1 kg             1 kmol     
                     ---------------   ---------------
          enthalpy       -2.5674e+05       -7.0947e+06  J
   internal energy        -3.464e+05       -9.5724e+06  J
           entropy            7240.5        2.0008e+05  J/K
    Gibbs function       -2.4144e+06       -6.6719e+07  J
 heat capacity c_p            1076.9             29758  J/K
 heat capacity c_v            775.99             21443  J/K

                      mass frac. Y      mole frac. X     chem. pot. / RT
                     ---------------   ---------------   ---------------
                O2           0.22014           0.19011           -26.334
               CH4          0.055187          0.095057           -54.877
                N2           0.72467           0.71483           -23.369


In [8]:
# Controllo sulla progress variable finale
T_c = 240 # [K]
reactants.TP = T_c, P_u
reactants.equilibrate("HP")
print(f"Final progress variable = {np.dot(reactants.Y, pv_weights): .11e}")

Final progress variable =  2.67938775818e-01
